# Role-Based Cybersecurity Risk Prediction
## Final Evidence Notebook - Logistic Regression Model v2.0.0

This notebook reproduces the model used by **CyberRisk Compass** and creates
the technical evidence required for the final-year project. Run the cells in
order. Keep the section heading and complete output visible in each screenshot.

**Research safeguards**
- Organization-separated train/test split
- Target-leakage prevention
- Preprocessing fitted only inside the model pipeline
- Group-aware cross-validation
- Final evaluation on previously unseen organizations
- Reproducible random seed: 42


## Evidence map

| Required evidence | Notebook section |
|---|---|
| Colab training stages | Sections 1-16 |
| Data preprocessing output | Section 5 |
| Organization-separated split | Section 4 |
| Model-comparison results | Section 7 |
| Final Logistic Regression metrics | Section 9 |
| Confusion matrix | Section 10 |
| 12 tests passed | Section 17 |
| Sample generated risk report | Section 18 |


## 1. Install and initialize the Colab environment


In [ ]:
!pip install -q openpyxl joblib seaborn reportlab pytest
print("Required packages installed successfully.")


In [ ]:
import hashlib
import json
import os
import random
import shutil
import subprocess
import sys
import urllib.request
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print("Environment initialized successfully.")
print(f"Python: {sys.version.split()[0]}")
print(f"pandas: {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"Random seed: {RANDOM_STATE}")


## 2. Load the research dataset


In [ ]:
DATASET_URL = (
    "https://raw.githubusercontent.com/Malshikafernando/CyberProject/"
    "main/Role_Based_Cybersecurity_Risk_Dataset_5500.xlsx"
)
DATASET_PATH = Path("Role_Based_Cybersecurity_Risk_Dataset_5500.xlsx")

if not DATASET_PATH.exists():
    urllib.request.urlretrieve(DATASET_URL, DATASET_PATH)

df = pd.read_excel(DATASET_PATH, sheet_name="Employee_Risk_Data", engine="openpyxl")
data_dictionary = pd.read_excel(DATASET_PATH, sheet_name="Data_Dictionary", engine="openpyxl")
recommendation_rules = pd.read_excel(
    DATASET_PATH, sheet_name="Recommendation_Rules", engine="openpyxl"
)

print(f"Loaded file: {DATASET_PATH.name}")
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Data dictionary rows: {len(data_dictionary):,}")
print(f"Recommendation rules: {len(recommendation_rules):,}")
display(df.head())


## 3. Data quality, schema audit and leakage prevention


In [ ]:
TARGET = "risk_level"
GROUP_COLUMN = "organization_id"
EXCLUDED_COLUMNS = [
    "record_id", "organization_id", "assessment_date", "data_source",
    "overall_risk_score", "role_risk_score", "risk_level",
    "primary_risk_category", "recommendation_priority",
    "recommended_action_codes", "training_focus_area",
]

required = set(EXCLUDED_COLUMNS + [GROUP_COLUMN, TARGET])
missing_required = sorted(required.difference(df.columns))
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

audit_summary = pd.DataFrame({
    "Measure": [
        "Rows", "Columns", "Duplicate rows", "Unique organizations",
        "Unique employee roles", "Total missing cells",
    ],
    "Value": [
        len(df), df.shape[1], df.duplicated().sum(),
        df[GROUP_COLUMN].nunique(), df["employee_role"].nunique(),
        int(df.isna().sum().sum()),
    ],
})
display(audit_summary)

X = df.drop(columns=EXCLUDED_COLUMNS).copy()
y = df[TARGET].astype("string").str.strip().str.title()
groups = df[GROUP_COLUMN].astype("string")

print(f"Predictor variables after leakage removal: {X.shape[1]}")
print(f"Excluded leakage/identifier columns: {len(EXCLUDED_COLUMNS)}")
print(f"Target classes: {sorted(y.unique())}")
display(y.value_counts().rename_axis("Risk level").to_frame("Records"))


## 4. Organization-separated train/test split


In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_index, test_index = next(splitter.split(X, y, groups))

X_train, X_test = X.iloc[train_index].copy(), X.iloc[test_index].copy()
y_train, y_test = y.iloc[train_index].copy(), y.iloc[test_index].copy()
train_groups = groups.iloc[train_index].copy()
test_groups = groups.iloc[test_index].copy()

organization_overlap = set(train_groups).intersection(set(test_groups))
assert len(organization_overlap) == 0, "Organization leakage detected."

split_summary = pd.DataFrame({
    "Partition": ["Training", "Testing"],
    "Records": [len(X_train), len(X_test)],
    "Organizations": [train_groups.nunique(), test_groups.nunique()],
})
display(split_summary)
print(f"Organization overlap: {len(organization_overlap)}")
print("PASS: No organization appears in both training and testing data.")


## 5. Data preprocessing and normalization pipeline


In [ ]:
numerical_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_features),
    ("categorical", categorical_pipeline, categorical_features),
])

preprocessing_summary = pd.DataFrame({
    "Feature type": ["Numerical", "Categorical", "Total raw inputs"],
    "Count": [len(numerical_features), len(categorical_features), X.shape[1]],
    "Processing": [
        "Median imputation + StandardScaler",
        "Most-frequent imputation + OneHotEncoder",
        "All operations inside the fitted pipeline",
    ],
})
display(preprocessing_summary)
print("PASS: Preprocessing pipeline created without fitting on test data.")


## 6. Baseline model


In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)
baseline_results = {
    "Accuracy": accuracy_score(y_test, baseline_predictions),
    "Balanced accuracy": balanced_accuracy_score(y_test, baseline_predictions),
    "Macro F1": f1_score(y_test, baseline_predictions, average="macro", zero_division=0),
}
display(pd.DataFrame([baseline_results], index=["Dummy baseline"]).round(4))


## 7. Compare candidate algorithms using organization-aware cross-validation


In [ ]:
candidate_models = {
    "logistic_regression": Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", LogisticRegression(
            max_iter=3000, class_weight="balanced", solver="saga",
            random_state=RANDOM_STATE,
        )),
    ]),
    "random_forest": Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", RandomForestClassifier(
            n_estimators=600, max_features="sqrt", min_samples_leaf=1,
            class_weight="balanced_subsample", random_state=RANDOM_STATE,
            n_jobs=1,
        )),
    ]),
    "extra_trees": Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", ExtraTreesClassifier(
            n_estimators=600, max_features="sqrt", min_samples_leaf=1,
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=1,
        )),
    ]),
}
print("Candidate pipelines created:", ", ".join(candidate_models))


In [ ]:
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "macro_f1": "f1_macro",
}
group_cv = GroupKFold(n_splits=5)
comparison_rows = []
cv_results = {}

for model_name, model in candidate_models.items():
    print(f"Evaluating {model_name}...")
    scores = cross_validate(
        model, X_train, y_train, groups=train_groups,
        cv=group_cv, scoring=scoring, n_jobs=-1, error_score="raise",
    )
    cv_results[model_name] = {
        metric: {
            "mean": round(float(scores[f"test_{metric}"].mean()), 6),
            "std": round(float(scores[f"test_{metric}"].std()), 6),
        }
        for metric in scoring
    }
    comparison_rows.append({
        "Model": model_name.replace("_", " ").title(),
        "Mean accuracy": scores["test_accuracy"].mean(),
        "Mean balanced accuracy": scores["test_balanced_accuracy"].mean(),
        "Mean macro F1": scores["test_macro_f1"].mean(),
        "Macro F1 std": scores["test_macro_f1"].std(),
    })

model_comparison = pd.DataFrame(comparison_rows).sort_values(
    "Mean macro F1", ascending=False
).reset_index(drop=True)
display(model_comparison.style.format({
    "Mean accuracy": "{:.4f}",
    "Mean balanced accuracy": "{:.4f}",
    "Mean macro F1": "{:.4f}",
    "Macro F1 std": "{:.4f}",
}))


## 8. Select and fit the best model


In [ ]:
selected_model_name = max(
    cv_results,
    key=lambda name: (
        cv_results[name]["macro_f1"]["mean"],
        cv_results[name]["balanced_accuracy"]["mean"],
    ),
)
best_model = clone(candidate_models[selected_model_name])
best_model.fit(X_train, y_train)

print(f"Selection metric: Mean macro F1")
print(f"Selected model: {selected_model_name}")
print(f"Winning CV macro F1: {cv_results[selected_model_name]['macro_f1']['mean']:.4f}")
assert selected_model_name == "logistic_regression"
print("PASS: Logistic Regression selected by group-aware cross-validation.")


## 9. Final Logistic Regression metrics on unseen organizations


In [ ]:
test_predictions = best_model.predict(X_test)
test_probabilities = best_model.predict_proba(X_test)

evaluation_metrics = {
    "Accuracy": accuracy_score(y_test, test_predictions),
    "Balanced accuracy": balanced_accuracy_score(y_test, test_predictions),
    "Macro F1": f1_score(y_test, test_predictions, average="macro", zero_division=0),
}
metric_table = pd.DataFrame({
    "Metric": list(evaluation_metrics),
    "Value": list(evaluation_metrics.values()),
})
display(metric_table.style.format({"Value": "{:.4f}"}))

class_order = ["Low", "Medium", "High"]
print("Detailed classification report:")
print(classification_report(
    y_test, test_predictions, labels=class_order, zero_division=0
))
assert evaluation_metrics["Macro F1"] > baseline_results["Macro F1"]
print("PASS: Final model outperforms the baseline on unseen organizations.")


## 10. Confusion matrix


In [ ]:
matrix = confusion_matrix(y_test, test_predictions, labels=class_order)
print("Rows = actual class; columns = predicted class")
display(pd.DataFrame(matrix, index=class_order, columns=class_order))

plt.figure(figsize=(8, 6))
sns.heatmap(
    matrix, annot=True, fmt="d", cmap="Blues", cbar=False,
    xticklabels=class_order, yticklabels=class_order,
)
plt.title("Confusion Matrix: Logistic Regression on Unseen Organizations")
plt.xlabel("Predicted Risk Level")
plt.ylabel("Actual Risk Level")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()


## 11. Role-level reliability analysis


In [ ]:
role_evaluation = X_test[["employee_role"]].copy()
role_evaluation["actual_risk"] = y_test.values
role_evaluation["predicted_risk"] = test_predictions

role_rows = []
for role, role_data in role_evaluation.groupby("employee_role"):
    role_rows.append({
        "Employee role": role,
        "Test records": len(role_data),
        "Accuracy": accuracy_score(
            role_data["actual_risk"], role_data["predicted_risk"]
        ),
        "Macro F1": f1_score(
            role_data["actual_risk"], role_data["predicted_risk"],
            average="macro", zero_division=0,
        ),
    })
role_performance = pd.DataFrame(role_rows).sort_values("Macro F1", ascending=False)
display(role_performance.style.format({"Accuracy": "{:.4f}", "Macro F1": "{:.4f}"}))


## 12. Logistic Regression coefficient evidence


In [ ]:
trained_preprocessor = best_model.named_steps["preprocessor"]
trained_classifier = best_model.named_steps["classifier"]
transformed_names = trained_preprocessor.get_feature_names_out()
mean_absolute_coefficient = np.abs(trained_classifier.coef_).mean(axis=0)

coefficient_evidence = pd.DataFrame({
    "Transformed feature": [
        name.replace("numerical__", "").replace("categorical__", "")
        for name in transformed_names
    ],
    "Mean absolute coefficient": mean_absolute_coefficient,
}).sort_values("Mean absolute coefficient", ascending=False)
display(coefficient_evidence.head(20).style.format({"Mean absolute coefficient": "{:.4f}"}))


## 13. Prediction confidence and error analysis


In [ ]:
prediction_details = X_test[["employee_role", "department"]].copy()
prediction_details["Actual risk"] = y_test.values
prediction_details["Predicted risk"] = test_predictions
prediction_details["Confidence"] = test_probabilities.max(axis=1)
prediction_details["Correct"] = (
    prediction_details["Actual risk"] == prediction_details["Predicted risk"]
)
print("Lowest-confidence predictions:")
display(prediction_details.sort_values("Confidence").head(15).style.format({"Confidence": "{:.4f}"}))


## 14. Build and save the deployment model package


In [ ]:
def friendly_label(name):
    replacements = {
        "Mfa": "MFA", "Vpn": "VPN", "Edr": "EDR",
        "12M": "12 Months", "30D": "30 Days",
    }
    label = name.replace("_", " ").title()
    for original, replacement in replacements.items():
        label = label.replace(original, replacement)
    return label

dictionary_lookup = data_dictionary.set_index("column_name")
field_schema = []
for name in X.columns:
    metadata = dictionary_lookup.loc[name]
    data_type = str(metadata["data_type"])
    field = {
        "name": name,
        "label": friendly_label(name),
        "group": str(metadata["category"]),
        "description": str(metadata["description"]),
    }
    if data_type == "Binary":
        field.update(
            input_kind="binary",
            options=[{"value": "1", "label": "Yes"}, {"value": "0", "label": "No"}],
            default=str(int(X[name].mode().iloc[0])),
        )
    elif name in categorical_features:
        field.update(
            input_kind="select",
            options=sorted(str(value) for value in X[name].dropna().unique()),
            default=str(X[name].mode().iloc[0]),
        )
    else:
        integer = pd.api.types.is_integer_dtype(X[name])
        median = float(X[name].median())
        field.update(
            input_kind="number",
            minimum=float(X[name].min()), maximum=float(X[name].max()),
            step=1 if integer else 0.1,
            default=int(round(median)) if integer else round(median, 1),
            numeric_type="int" if integer else "float",
        )
    field_schema.append(field)

final_model = clone(candidate_models[selected_model_name]).fit(X, y)
role_department_map = (
    df.groupby("employee_role")["department"].first().astype(str).to_dict()
)
MODEL_FILENAME = "role_based_cybersecurity_risk_model.pkl"
REPORT_FILENAME = "role_based_model_report.json"

model_package = {
    "model_name": "Role-Based Cybersecurity Risk Prediction Model",
    "model_version": "2.0.0",
    "pipeline": final_model,
    "target_column": TARGET,
    "class_labels": final_model.named_steps["classifier"].classes_.tolist(),
    "feature_columns": X.columns.tolist(),
    "numerical_features": numerical_features,
    "categorical_features": categorical_features,
    "excluded_columns": EXCLUDED_COLUMNS,
    "field_schema": field_schema,
    "role_department_map": role_department_map,
    "recommendation_rules": recommendation_rules.replace({np.nan: None}).to_dict(orient="records"),
    "selected_model": selected_model_name,
    "evaluation_metrics": evaluation_metrics,
    "training_records_final": int(len(X)),
    "holdout_records": int(len(X_test)),
    "training_organizations": int(train_groups.nunique()),
    "testing_organizations": int(test_groups.nunique()),
    "random_state": RANDOM_STATE,
    "trained_at_utc": datetime.now(timezone.utc).isoformat(),
    "data_notice": "Trained on a synthetic academic research dataset.",
}
joblib.dump(model_package, MODEL_FILENAME, compress=3)
model_sha256 = hashlib.sha256(Path(MODEL_FILENAME).read_bytes()).hexdigest()

print(f"Model saved: {MODEL_FILENAME}")
print(f"Model version: {model_package['model_version']}")
print(f"Selected model: {model_package['selected_model']}")
print(f"Final training records: {model_package['training_records_final']:,}")
print(f"SHA-256: {model_sha256}")


## 15. Export the machine-readable training report


In [ ]:
report = {
    "model_name": model_package["model_name"],
    "model_version": model_package["model_version"],
    "dataset_rows": int(len(df)),
    "input_features": int(X.shape[1]),
    "target_distribution": {k: int(v) for k, v in y.value_counts().items()},
    "organization_group_split": {
        "training_organizations": int(train_groups.nunique()),
        "testing_organizations": int(test_groups.nunique()),
        "overlap": 0,
    },
    "selected_model": selected_model_name,
    "selection_metric": "macro_f1",
    "cross_validation_on_training_organizations": cv_results,
    "unseen_organization_holdout": {
        "accuracy": round(float(evaluation_metrics["Accuracy"]), 6),
        "balanced_accuracy": round(float(evaluation_metrics["Balanced accuracy"]), 6),
        "macro_f1": round(float(evaluation_metrics["Macro F1"]), 6),
        "confusion_matrix_labels": class_order,
        "confusion_matrix": matrix.tolist(),
        "classification_report": classification_report(
            y_test, test_predictions, labels=class_order,
            output_dict=True, zero_division=0,
        ),
    },
    "model_sha256": model_sha256,
    "library_versions": {
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "joblib": joblib.__version__,
    },
    "trained_at_utc": model_package["trained_at_utc"],
    "data_notice": model_package["data_notice"],
}
Path(REPORT_FILENAME).write_text(json.dumps(report, indent=2), encoding="utf-8")
print(f"Created {REPORT_FILENAME}")
display(pd.DataFrame([
    {"Evidence": "Selected model", "Result": selected_model_name},
    {"Evidence": "Holdout accuracy", "Result": f"{evaluation_metrics['Accuracy']:.4f}"},
    {"Evidence": "Holdout balanced accuracy", "Result": f"{evaluation_metrics['Balanced accuracy']:.4f}"},
    {"Evidence": "Holdout macro F1", "Result": f"{evaluation_metrics['Macro F1']:.4f}"},
    {"Evidence": "Organization overlap", "Result": "0"},
]))


## 16. Reload and validate the saved model


In [ ]:
loaded_package = joblib.load(MODEL_FILENAME)
sample_input = X_test.iloc[[0]].copy()
saved_prediction = loaded_package["pipeline"].predict(sample_input)[0]
saved_probabilities = loaded_package["pipeline"].predict_proba(sample_input)[0]

assert sample_input.columns.tolist() == loaded_package["feature_columns"]
assert loaded_package["selected_model"] == "logistic_regression"
print("PASS: Saved model validation completed successfully.")
print(f"Sample employee role: {sample_input['employee_role'].iloc[0]}")
print(f"Predicted risk: {saved_prediction}")
display(pd.DataFrame({
    "Risk level": loaded_package["class_labels"],
    "Probability": saved_probabilities,
}).sort_values("Probability", ascending=False).style.format({"Probability": "{:.4f}"}))


## 17. Automated application tests - required screenshot


In [ ]:
REPO_DIR = Path("/content/CyberProject")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run([
    "git", "clone", "-q",
    "https://github.com/Malshikafernando/CyberProject.git",
    str(REPO_DIR),
], check=True)
test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=REPO_DIR,
    text=True,
    capture_output=True,
)
print("Command: python -m pytest -q")
print(test_result.stdout)
if test_result.stderr:
    print(test_result.stderr)
assert test_result.returncode == 0, "Application tests failed."


## 18. Generate a sample PDF risk report - required screenshot


In [ ]:
import importlib.util

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

app_spec = importlib.util.spec_from_file_location(
    "cyberproject_live_app", REPO_DIR / "app.py"
)
app_module = importlib.util.module_from_spec(app_spec)
app_spec.loader.exec_module(app_module)

deployed_package = app_module.load_model_package()
report_payload = app_module.default_form_values(deployed_package)
report_payload["assessment_reference"] = "COLAB-EVIDENCE-001"

with app_module.app.test_client() as client:
    prediction_response = client.post("/predict", data=report_payload)
    assert prediction_response.status_code == 302
    pdf_response = client.get("/download-report")
    assert pdf_response.status_code == 200
    SAMPLE_REPORT = Path("cyberrisk-compass-sample-report.pdf")
    SAMPLE_REPORT.write_bytes(pdf_response.data)

print(f"Generated report: {SAMPLE_REPORT}")
print(f"PDF size: {SAMPLE_REPORT.stat().st_size:,} bytes")
print(f"PDF signature valid: {SAMPLE_REPORT.read_bytes().startswith(b'%PDF-')}")
print("Assessment reference: COLAB-EVIDENCE-001")

from google.colab import files
files.download(str(SAMPLE_REPORT))


## Screenshot checklist

Capture these outputs with the notebook name and section heading visible:

1. Section 2 - dataset loaded successfully
2. Section 4 - 176/44 organizations and overlap 0
3. Section 5 - preprocessing summary
4. Section 7 - model comparison table
5. Section 9 - final Logistic Regression metrics and classification report
6. Section 10 - confusion matrix
7. Section 17 - `12 passed`
8. Section 18 - successful PDF generation, then open the downloaded PDF for its screenshot

Suggested filenames: `E01_dataset.png` through `E08_sample_report.png`.
